In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)


In [3]:
# Loading the cleaned dataset

df = pd.read_csv("../data/cleaned/deforestation_cleaned.csv")

print("Dataset Shape:", df.shape)

print("Missing Values:")
print(df.isna().sum())

Dataset Shape: (1397, 12)
Missing Values:
Record_ID             0
Observation_Date      0
Region                0
District              0
Forest_Type           0
Forest_Area_ha        0
Tree_Cover_Loss_ha    0
Annual_Rainfall_mm    0
Population_Density    0
Fire_Incidents        0
Illegal_Logging       0
Deforestation_Risk    0
dtype: int64


In [4]:
# Drop unnecessary columns
df.drop(columns=["Record_ID"], inplace=True)

# Standardize the date column
df['Observation_Date'] = pd.to_datetime(df['Observation_Date'], errors='coerce')

df["Year"] = df["Observation_Date"].dt.year
df["Month"] = df["Observation_Date"].dt.month
df["Day"] = df["Observation_Date"].dt.day

# Converting numeric columns stored as strings
numeric_columns = [
    "Annual_Rainfall_mm",
    "Population_Density"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check for missing values created during conversion
print("\nMissing Values")
print(df.isna().sum())

print("\nUnique Values")
print(df.nunique())


Missing Values
Observation_Date      1
Region                0
District              0
Forest_Type           0
Forest_Area_ha        0
Tree_Cover_Loss_ha    0
Annual_Rainfall_mm    1
Population_Density    1
Fire_Incidents        0
Illegal_Logging       0
Deforestation_Risk    0
Year                  1
Month                 1
Day                   1
dtype: int64

Unique Values
Observation_Date       889
Region                   4
District                 6
Forest_Type              6
Forest_Area_ha        1384
Tree_Cover_Loss_ha    1363
Annual_Rainfall_mm    1336
Population_Density     682
Fire_Incidents          26
Illegal_Logging          3
Deforestation_Risk       3
Year                     4
Month                   12
Day                     31
dtype: int64


In [5]:
# Convert 'Illegal_Logging' from 3-way categorical to binary (0/1)
df["Illegal_Logging_Flag"] = df["Illegal_Logging"].map({
    "Yes": 1,
    "No": 0
})

print("\nMissing Values:")
print(df.isna().sum())


Missing Values:
Observation_Date        1
Region                  0
District                0
Forest_Type             0
Forest_Area_ha          0
Tree_Cover_Loss_ha      0
Annual_Rainfall_mm      1
Population_Density      1
Fire_Incidents          0
Illegal_Logging         0
Deforestation_Risk      0
Year                    1
Month                   1
Day                     1
Illegal_Logging_Flag    1
dtype: int64


In [6]:
# Drop All null record
df = df.dropna()

print("\nMissing Values:")
print(df.isna().sum())


Missing Values:
Observation_Date        0
Region                  0
District                0
Forest_Type             0
Forest_Area_ha          0
Tree_Cover_Loss_ha      0
Annual_Rainfall_mm      0
Population_Density      0
Fire_Incidents          0
Illegal_Logging         0
Deforestation_Risk      0
Year                    0
Month                   0
Day                     0
Illegal_Logging_Flag    0
dtype: int64


In [7]:
# One-Hot Encoding
df = pd.get_dummies(
    df,
    columns=["Region", "District", "Forest_Type"],
    dtype=int
)

# Drop new null records
df.dropna(axis=0, inplace=True)


# Label Encoding
categorical_columns = "Deforestation_Risk"

encoder = LabelEncoder()
df[categorical_columns] = encoder.fit_transform(df[categorical_columns])

# Save all encoders
# joblib.dump(encoder, "../src/label_encoder.pkl")


# Human pressure relative to forest size
df["Population_Pressure"] = (
    df["Population_Density"] /
    df["Forest_Area_ha"]
)

# Combined impact of fire and illegal logging
df["Fire_Logging_Impact"] = (
    df["Fire_Incidents"] *
    df["Illegal_Logging_Flag"]
)

print("New Features Created Successfully!")

print(df.head())


New Features Created Successfully!
  Observation_Date  Forest_Area_ha  Tree_Cover_Loss_ha  Annual_Rainfall_mm  \
0       2024-06-27         4179.46              235.27              1358.1   
1       2021-04-30         2072.03               14.36              1605.3   
2       2022-01-20         7417.11              543.23              1471.4   
3       2021-07-17         5260.01              281.86              1609.9   
4       2023-07-07         9793.89              575.77              1353.5   

   Population_Density  Fire_Incidents Illegal_Logging  Deforestation_Risk  \
0               657.0              16             Yes                   0   
1               266.0              25             Yes                   0   
2               245.0              16             Yes                   0   
3               434.0              11             Yes                   0   
4               647.0               6              No                   0   

     Year  Month   Day  Illegal_L

In [8]:
# Saving the feature engineered dataset

# output_file = "../data/engineered/deforestation_features.csv"

# df.to_csv(output_file, index=False)

# print(f"Dataset saved successfully to {output_file}")